# Exercitiu Laborator 2 - Formula lui Bayes

## Datele problemei
- Probabilitatea ca o persoana sa fie bolnava: $P(B) = 0.01$
- Probabilitatea ca o persoana sa fie sanatoasa: $P(\neg B) = 1 - 0.01 = 0.99$
- Sensibilitate (True Positive Rate): $P(+ \mid B) = 0.95$
- Specificitate (True Negative Rate): $P(- \mid \neg B) = 0.90$
- Rata de fals-pozitiv: $P(+ \mid \neg B) = 1 - 0.90 = 0.10$

## Punctul a) Calculul probabilitatii si explicatia rezultatului

Conform formulei lui Bayes:
$$P(B \mid +) = \frac{P(+ \mid B) \cdot P(B)}{P(+)}$$

Unde probabilitatea totala ca testul sa iasa pozitiv este:
$$P(+) = P(+ \mid B) \cdot P(B) + P(+ \mid \neg B) \cdot P(\neg B)$$
$$P(+) = 0.95 \cdot 0.01 + 0.10 \cdot 0.99 = 0.0095 + 0.099 = 0.1085$$

$$P(B \mid +) = \frac{0.0095}{0.1085} = \frac{19}{217} \approx 0.087557 \quad (\approx 8.76\%)\$$

### Explicatia rezultatului:
Desi testul are o acuratete ridicata (95% sensibilitate si 90% specificitate), probabilitatea ca o persoana pozitiva sa fie cu adevarat bolnava este sub 9% (fenomenul base rate fallacy).

Deoarece boala este rara in populatie (1%), marea majoritate a oamenilor sunt sanatosi (99%). Rata de fals-pozitiv de 10%, aplicata pe aceasta masa mare de 99% din populatie, genereaza aproximativ 9.9% rezultate fals-pozitive in intreaga populatie. Aceasta cantitate este de peste 10 ori mai mare decat cei 0.95% bolnavi depistati corect pozitiv.

## Punctul b) Modelare in Python si comparatie cu rezultatul teoretic

In [1]:
import numpy as np

p_b = 0.01
sensibilitate = 0.95
specificitate = 0.90
n_simulari = 10_000_000

p_pos_given_not_b = 1.0 - specificitate
p_pos = (sensibilitate * p_b) + (p_pos_given_not_b * (1.0 - p_b))
p_b_given_pos_teoretic = (sensibilitate * p_b) / p_pos

bolnav = np.random.rand(n_simulari) < p_b

test_pozitiv = np.zeros(n_simulari, dtype=bool)
test_pozitiv[bolnav] = np.random.rand(np.sum(bolnav)) < sensibilitate
test_pozitiv[~bolnav] = np.random.rand(np.sum(~bolnav)) < (1.0 - specificitate)

p_b_given_pos_simulat = np.sum(bolnav & test_pozitiv) / np.sum(test_pozitiv)

print(f"Probabilitate teoretica : {p_b_given_pos_teoretic:.6f} ({p_b_given_pos_teoretic * 100:.2f}%)")
print(f"Probabilitate simulata  : {p_b_given_pos_simulat:.6f} ({p_b_given_pos_simulat * 100:.2f}%)")
print(f"Diferenta absoluta      : {abs(p_b_given_pos_teoretic - p_b_given_pos_simulat):.6e}")

Probabilitate teoretica : 0.087558 (8.76%)
Probabilitate simulata  : 0.087464 (8.75%)
Diferenta absoluta      : 9.386950e-05


## Punctul c) Specificitatea minima pentru ca P(B | +) sa fie cel putin 50%

Notam specificitatea cu $S_p = P(- \mid \neg B)$. Rata fals-pozitiva este $P(+ \mid \neg B) = 1 - S_p$.

$$P(B \mid +) \ge 0.5$$

$$\frac{P(+ \mid B) \cdot P(B)}{P(+ \mid B) \cdot P(B) + (1 - S_p) \cdot P(\neg B)} \ge 0.5$$

$$P(+ \mid B) \cdot P(B) \ge (1 - S_p) \cdot P(\neg B)$$

$$0.95 \cdot 0.01 \ge (1 - S_p) \cdot 0.99$$
$$0.0095 \ge (1 - S_p) \cdot 0.99$$
$$1 - S_p \le \frac{0.0095}{0.99} = \frac{95}{9900} = \frac{19}{1980} \approx 0.009596$$
$$S_p \ge 1 - \frac{19}{1980} = \frac{1961}{1980} \approx 0.990404$$

Specificitatea minima necesara este de aproximativ **99.04%** (sau $\frac{1961}{1980}$).

In [2]:
p_b = 0.01
sensibilitate = 0.95
target_p = 0.50

fpr_max = (sensibilitate * p_b * (1 - target_p)) / (target_p * (1 - p_b))
sp_min = 1.0 - fpr_max

print(f"Specificitatea minima teoretica: {sp_min:.6f} ({sp_min * 100:.4f}%)")

p_pos_check = (sensibilitate * p_b) + ((1 - sp_min) * (1 - p_b))
p_b_given_pos_check = (sensibilitate * p_b) / p_pos_check
print(f"P(B | +) la aceasta specificitate: {p_b_given_pos_check * 100:.2f}%")

Specificitatea minima teoretica: 0.990404 (99.0404%)
P(B | +) la aceasta specificitate: 50.00%
